<a href="https://colab.research.google.com/github/goksungurel/CE_477/blob/main/section2_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

URL_2024 = "https://acikveri.bizizmir.com/dataset/95bec47b-a29c-4ab0-a070-434334588da8/resource/23bfeebe-1370-49da-aed0-5573cf3dd612/download/2024-yili-yangin-mudahale-cizelgesi.xlsx$0"
URL_2025 = "https://acikveri.bizizmir.com/dataset/95bec47b-a29c-4ab0-a070-434334588da8/resource/d9bfb046-f67b-4fb3-bc62-43068895c19f/download/2025-yili-yangin-mudahale-cizelgesi.xlsx$0"

d24 = pd.read_excel(URL_2024); d24["YIL"] = 2024
d25 = pd.read_excel(URL_2025); d25["YIL"] = 2025
df = pd.concat([d24, d25], ignore_index=True)
print(df.shape)
df.head()

Setup (rename + type fixes)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, os
os.makedirs("figures", exist_ok=True)

# English column names
df = df.rename(columns={
    "TARIH": "date", "YANGIN_TURU": "fire_type", "YANGIN_SEBEBİ": "fire_cause",
    "YANGIN_SONUCU": "fire_result", "YAPI_SEKLI": "building_type",
    "ERKEK_VEFAT": "male_deaths", "KADIN_VEFAT": "female_deaths", "ITFAIYECI_VEFAT": "firefighter_deaths",
    "ERKEK_YARALI": "male_injured", "KADIN_YARALI": "female_injured", "ITFAIYECI_YARALI": "firefighter_injured",
    "BUYUK_BAS_VEFAT": "cattle_deaths", "KUCUK_BAS_VEFAT": "sheep_goat_deaths",
    "KUMES_HAYVANI_VEFAT": "poultry_deaths", "EVCIL_HAYVAN_VEFAT": "pet_deaths",
    "DIGER HAYVAN": "other_animal_deaths", "VARIS_SURESI (DAK.)": "arrival_time",
    "KULLANILAN_KOPUK_MIKTARI (KG)": "foam_kg", "KULLANILAN_SU_MIKTARI (m³)": "water_m3",
    "KULLANILAN_KURU_KIMYEVİ_TOZ_MIKTARI (KG)": "dry_powder_kg",
    "EKIPLERİN_CIKIS_SAATI": "dispatch_time", "ILCE": "district",
    "ADRES_BOLGESI": "region", "YIL": "year"})

# type fixes + derived attributes
df["arrival_min"] = pd.to_timedelta(df["arrival_time"] + ":00").dt.total_seconds() / 60
df["dispatch_hour"] = df["dispatch_time"].str[:2].astype(int)
df["other_animal_deaths"] = pd.to_numeric(df["other_animal_deaths"], errors="coerce")
df["month"] = df["date"].dt.month

# English category labels
df["fire_result"] = df["fire_result"].map({
    "BAŞLANGIÇTA SÖNDÜRÜLEN": "Extinguished early",
    "KISMEN YANARAK KURTARILAN": "Partially burned",
    "TAMAMEN YANAN": "Completely burned"})
result_order = ["Extinguished early", "Partially burned", "Completely burned"]
df["fire_result"] = pd.Categorical(df["fire_result"], categories=result_order, ordered=True)
df["region"] = df["region"].map({"KENT MERKEZİ": "Urban", "KIRSAL BÖLGE": "Rural", "İL DIŞI": "Outside province"})
df["fire_type"] = df["fire_type"].replace({
    "OT": "Grass", "ÇÖP": "Garbage", "BİNA": "Building", "ARAÇ": "Vehicle", "İŞYERİ": "Workplace",
    "AĞAÇ": "Tree", "KARGILIK": "Reed bed", "ORMAN/FİDANLIK": "Forest/nursery",
    "BAĞ/BAHÇE": "Vineyard/garden", "TRAFO": "Transformer",
    "FABRİKA/ATÖLYE/İMALATHANE": "Factory/workshop", "KABLO/DİREK-ELEKT/TEL VB": "Cable/pole",
    "BARAKA/KULÜBE/YAŞAM KONTEYNERİ": "Shack/container", "ZEYTİNLİK": "Olive grove"})
df["fire_cause"] = df["fire_cause"].replace({
    "SİGARA/KİBRİT": "Cigarette/match", "AÇIK ATEŞ": "Open fire",
    "ELEKT. KISA DEV./ELEKT. KONTAĞI": "Electrical short circuit",
    "KIVILCIM(DİĞER)": "Spark (other)", "KIVILCIM(BACA)": "Spark (chimney)",
    "ORMAN İTF. TESPİT EDECEK": "To be determined by forestry", "ŞÜPHELİ": "Suspicious",
    "ISINMA/SÜRTÜNME": "Heating/friction", "OCAK ÜSTÜ YEMEK UNUTMA": "Unattended cooking",
    "SİRAYET": "Spread from nearby fire"})
df["building_type"] = df["building_type"].replace({
    "BETONARME": "Reinforced concrete", "DİĞER": "Other", "KAGİR": "Masonry", "ÇELİK": "Steel",
    "AHŞAP": "Timber", "PREFABRİK": "Prefabricated", "BAĞDADİ(TAŞ YAPI)": "Bagdadi (timber-frame)"})
print(df.shape)
df.dtypes

In [ ]:
# translate the remaining fire types and causes
df["fire_type"] = df["fire_type"].replace({
    "EKİN/SAMAN": "Crop/straw", "MAKİLİK/ÇALI": "Maquis/shrubland", "ANIZ": "Stubble",
    "DİĞER": "Other", "HURDA-AÇIK ALANDA": "Scrap (open area)", "LASTİK": "Tyre",
    "SİGORTA PANO-BİNADAN BAĞIMSIZ": "Electrical panel (detached)", "MOTOSİKLET": "Motorcycle",
    "DAM/AHIR/AĞIL/AMBAR": "Barn/stable/shed", "ODUNLUK/KÖMÜRLÜK": "Woodshed/coal store",
    "ODUN/KÖMÜR-AÇIK ALANDA": "Wood/coal (open area)", "AÇIK ALAN DEPO": "Open-air storage",
    "KERESTE/PALET VB": "Timber/pallets", "GÜBRE": "Manure/fertiliser", "GEMİ/TEKNE": "Ship/boat",
    "HURDALIK": "Scrapyard", "İNŞAAT ALANI": "Construction site", "İŞ MAKİNESİ": "Construction machinery",
    "PARK": "Park", "SERA": "Greenhouse", "KÜMES": "Poultry house",
    "REKLAM/İŞYERİ TABELASI/PANOSU VB": "Billboard/signboard", "ARAÇ (ELEKTRİKLİ)": "Electric vehicle",
    "ARAÇ-TEHLİKELİ MAD TAŞIYAN": "Hazmat vehicle", "MOTOSİKLET (ELEKTRİKLİ)": "Electric motorcycle",
    "PETROL İSTASYONU": "Petrol station", "TREN": "Train"})
df["fire_cause"] = df["fire_cause"].replace({
    "PARLAMA (YAĞ/KİMYASAL MAD.)": "Flash fire (oil/chemical)", "PARLAMA (AKARYAKIT MADDESİ)": "Flash fire (fuel)",
    "İHMAL/DİKKATSİZLİK": "Negligence", "DİĞER": "Other",
    "ÇOCUKLARIN ATEŞLE OYNAMASI": "Children playing with fire", "KASIT": "Intentional",
    "KIZIŞMA": "Spontaneous heating", "LPG/DOĞALGAZ": "LPG/natural gas",
    "KONDÜKSİYON (TEMAS İLE ISI TRANSFERİ)": "Conduction (contact heat transfer)",
    "DOĞAL OLAYLAR (YILDIRIM/GÜNEŞ VB)": "Natural causes (lightning/sun)",
    "KUNDAKLAMA/SABOTAJ": "Arson/sabotage", "KAZA SONUCU": "Accident", "HARARET": "Engine overheating",
    "KİMYASAL MADDE REAKSİYONU": "Chemical reaction",
    "RADYASYON (IŞINIM İLE ISI TRANSFERİ)": "Radiation (radiant heat transfer)",
    "KONVEKSİYON (HAVA İLE ISI TRANSFERİ)": "Convection (air heat transfer)",
    "OCAK/SOBA/KALORİFER KAZANI": "Stove/heater/boiler",
    "KONTROLLÜ ATEŞİN YANGINA DÖNÜŞMESİ": "Controlled burn got out of control",
    "ELEKT. KISA DEV. (TERMAL KAÇAK)": "Electrical (thermal runaway)", "TESPİT EDİLEMEDİ": "Undetermined",
    "İNFİLAK": "Explosion", "STATİK ELEKTRİK": "Static electricity", "PATLAYICI MADDE": "Explosive material"})
print(df["fire_type"].unique())

Missing values


In [ ]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_%"] = (missing["missing_count"] / len(df) * 100).round(2)
print(missing)
# building_type is only filled for structure fires:
print(df[df["building_type"].notna()]["fire_type"].value_counts())

Box plots


In [ ]:
numeric = ["arrival_min", "water_m3", "foam_kg", "dry_powder_kg"]
titles = ["Arrival time (min)", "Water used (m³)", "Foam used (kg)", "Dry chemical powder (kg)"]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col, t in zip(axes, numeric, titles):
    sns.boxplot(y=np.log10(1 + df[col].dropna()), ax=ax)
    ax.set_title(t); ax.set_ylabel("log10(1 + value)")
plt.tight_layout(); plt.savefig("figures/boxplot.png", dpi=300); plt.show()
print(df[numeric].describe().round(2))

Histograms (ordinal)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["fire_result"].value_counts(sort=False).plot(kind="bar", ax=axes[0], logy=True)
axes[0].set_title("Fire result (log scale)"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=0)
axes[1].hist(df["dispatch_hour"], bins=24, edgecolor="white")
axes[1].set_title("Dispatch hour"); axes[1].set_xlabel("hour of day")
axes[2].hist(df["month"], bins=12, edgecolor="white")
axes[2].set_title("Month"); axes[2].set_xlabel("month")
for ax in axes: ax.set_ylabel("number of fires")
plt.tight_layout(); plt.savefig("figures/histogram.png", dpi=300); plt.show()

Mode/median

In [ ]:
categorical = ["fire_type", "fire_cause", "fire_result", "building_type", "district", "region"]
for c in categorical:
    print(f"{c}: mode = {df[c].mode()[0]}  ({df[c].value_counts().iloc[0]} records)")
print("fire_result median:", result_order[int(df["fire_result"].cat.codes.median())])

Bar + stacked bar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["fire_type"].value_counts().head(10).plot(kind="barh", ax=axes[0]); axes[0].invert_yaxis()
axes[0].set_title("Top 10 fire types"); axes[0].set_xlabel("number of fires"); axes[0].set_ylabel("")
df["fire_cause"].value_counts().head(10).plot(kind="barh", ax=axes[1], color="tab:orange"); axes[1].invert_yaxis()
axes[1].set_title("Top 10 fire causes"); axes[1].set_xlabel("number of fires"); axes[1].set_ylabel("")
plt.tight_layout(); plt.savefig("figures/bar.png", dpi=300); plt.show()

top10 = df["fire_type"].value_counts().head(10).index
table = pd.crosstab(df["fire_type"], df["fire_result"], normalize="index").loc[top10] * 100
table.plot(kind="barh", stacked=True, figsize=(9, 5))
plt.xlabel("% of fires"); plt.ylabel(""); plt.title("Fire result by fire type")
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=3)
plt.tight_layout(); plt.savefig("figures/stacked_bar.png", dpi=300); plt.show()
print(table.round(1))

Scatter plots

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="arrival_min", y=np.log10(1 + df["water_m3"]),
                hue="fire_result", alpha=0.4, s=12)
plt.xlim(0, 60); plt.xlabel("Arrival time (min)"); plt.ylabel("log10(1 + water m³)")
plt.title("Arrival time vs water used")
plt.savefig("figures/scatter.png", dpi=300, bbox_inches="tight"); plt.show()

dist = df.groupby("district").agg(
    median_arrival=("arrival_min", "median"),
    damaged_pct=("fire_result", lambda x: (x != "Extinguished early").mean() * 100),
    n_fires=("date", "size"))
plt.figure(figsize=(8, 5))
plt.scatter(dist["median_arrival"], dist["damaged_pct"], s=dist["n_fires"] / 10, alpha=0.6)
for name, row in dist.iterrows():
    plt.annotate(name, (row["median_arrival"], row["damaged_pct"]), fontsize=7)
plt.xlabel("Median arrival time (min)"); plt.ylabel("% partially/completely burned")
plt.title("Districts: arrival time vs damage rate (size = number of fires)")
plt.savefig("figures/scatter_district.png", dpi=300, bbox_inches="tight"); plt.show()

Line plots

In [ ]:
daily = df.groupby("date").size()
plt.figure(figsize=(13, 4))
plt.plot(daily, color="lightgray", label="daily")
plt.plot(daily.rolling(7).mean(), label="7-day moving average")
plt.ylabel("number of fires"); plt.title("Daily fire responses (2024–2025)"); plt.legend()
plt.savefig("figures/line_daily.png", dpi=300, bbox_inches="tight"); plt.show()
print(daily.sort_values(ascending=False).head())

monthly = df.groupby(["year", "month"]).size().unstack(0)
monthly.plot(marker="o", figsize=(9, 4))
plt.xticks(range(1, 13)); plt.xlabel("month"); plt.ylabel("number of fires")
plt.title("Monthly fires: 2024 vs 2025")
plt.savefig("figures/line_monthly.png", dpi=300, bbox_inches="tight"); plt.show()